# 07 · Classify images from your own folders

**Level:** intermediate. **Before this:** CNNs, augmentation, and validation.
**Practical outcome:** build an `ImageFolder` pipeline with class checks, stratified
splits, duplicate-content leakage checks, a trained classifier, and a reusable artifact.

This notebook runs without your personal dataset. Its default data is an explicitly
**synthetic** set of colored circles, squares, and triangles generated locally with
Pillow. It teaches the workflow; synthetic accuracy does not establish performance
on photographs. Point `DATASET_ROOT` at your own folders when you are ready.


## Step 1 · Configure the session

Run from a fresh kernel. The default is a short demonstration. All generated images
stay inside `data/generated_shapes_v1`; your own files are only read. On Windows,
the course loader uses zero worker processes to avoid notebook multiprocessing issues.


In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in the project root or in notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the complete course folder; course_utils.py is required.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import torch
import course_utils as cu

DEVICE = cu.configure(seed=42)
SMOKE = cu.SMOKE  # COURSE_SMOKE=1: validation only; too small to judge model quality.
QUICK = True     # Set False for a longer learning experiment, then Restart Kernel and Run All.


In [ ]:
import hashlib
import json
from collections import defaultdict
from PIL import Image, ImageDraw
from torchvision import datasets, transforms
from torch.utils.data import Subset
from sklearn.model_selection import train_test_split
import pandas as pd

cu.seed_everything(42)
EPOCHS = 1 if SMOKE else (3 if QUICK else 10)
BATCH_SIZE = 32
GENERATED_ROOT = cu.DATA_DIR / "generated_shapes_v1"
DATASET_ROOT = GENERATED_ROOT  # Change to Path(r"D:\my_images") for your own classes.


## Step 2 · Understand the required folder layout

`ImageFolder` derives labels from class directory names. Put original images in
folders like this; this notebook creates index-based splits without moving files:

```text
my_images/
  cat/    photo_001.jpg, photo_002.jpg, ...
  dog/    photo_101.jpg, photo_102.jpg, ...
  horse/  photo_201.jpg, photo_202.jpg, ...
```

Use at least ten distinct images per class for this tiny teaching split, and many
more for a useful real classifier. Do not mix an existing `train/val/test` directory
tree into this layout. If images belong to the same patient, animal, product, video,
or capture session, split by that group **before** augmentation; random image splits
can leak related images even when file content differs.

For the default shapes, every class uses the same background-color range and the
same foreground-color range; colors vary per image rather than encoding the class.
The generator is idempotent: it skips existing filenames. Editing its seed or colors
and rerunning does **not** modify images already on disk. To study a changed background,
use the separate, in-memory shift dataset in the final exercise and keep the trained
model fixed. A newly generated source dataset would need an unused versioned folder.


In [ ]:
def generate_shapes(root, per_class=160, seed=2026):
    # This helper may write only to the dedicated generated-data directory.
    root = Path(root).resolve()
    assert root == GENERATED_ROOT.resolve(), "Generator is restricted to its dedicated folder."
    names = ["circle", "square", "triangle"]
    created = 0
    for class_id, name in enumerate(names):
        folder = root / name
        folder.mkdir(parents=True, exist_ok=True)
        for i in range(per_class):
            path = folder / f"shape_{i:04d}.png"
            if path.exists():
                continue  # Idempotent: never overwrite an existing image.
            rng = np.random.default_rng(seed + class_id * 10000 + i)
            background = tuple(int(v) for v in rng.integers(215, 256, size=3))
            foreground = tuple(int(v) for v in rng.integers(20, 175, size=3))
            image = Image.new("RGB", (64, 64), background)
            draw = ImageDraw.Draw(image)
            size = int(rng.integers(22, 43))
            x, y = [int(v) for v in rng.integers(4, 60 - size, size=2)]
            box = (x, y, x + size, y + size)
            if name == "circle":
                draw.ellipse(box, fill=foreground)
            elif name == "square":
                draw.rectangle(box, fill=foreground)
            else:
                draw.polygon([(x + size // 2, y), (x, y + size), (x + size, y + size)],
                             fill=foreground)
            image.save(path)
            created += 1
    return created

if Path(DATASET_ROOT).resolve() == GENERATED_ROOT.resolve():
    print("New synthetic images:", generate_shapes(GENERATED_ROOT))
else:
    print("Using your image folders; no images will be generated here.")
assert Path(DATASET_ROOT).is_dir(), f"Dataset folder does not exist: {DATASET_ROOT}"
print("Dataset root:", Path(DATASET_ROOT).resolve())


## Step 3 · Inspect the label map and class balance

Label IDs follow folder-name ordering. Never assume that class `0` means your first
handwritten label. Save and reuse `class_to_idx` at inference. Inspect the counts to
spot missing classes or strong imbalance before training.


In [ ]:
catalog = datasets.ImageFolder(DATASET_ROOT)
class_names = catalog.classes
labels = np.asarray(catalog.targets)
counts = pd.Series(labels).value_counts().sort_index()
assert len(class_names) >= 2, "Classification needs at least two class folders."
assert counts.min() >= 10, "Use at least 10 distinct images per class for this demonstration."
print("Class map:", catalog.class_to_idx)
count_table = pd.DataFrame({"class": class_names, "images": counts.values})
display(count_table)
count_table.plot.bar(x="class", y="images", legend=False, title="Images per class")
plt.ylabel("Count")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## Step 4 · Validate files and fingerprint decoded pixels

An image extension does not guarantee that the image is readable. Open every image
once and hash its RGB pixels plus dimensions. This catches identical decoded images
even if filenames or metadata differ. It does **not** catch resized, recompressed,
cropped, or nearly identical images; group-aware review is still needed for those.
Large real datasets should run this audit once and cache the resulting manifest.


In [ ]:
def pixel_fingerprint(path):
    with Image.open(path) as source:
        rgb = source.convert("RGB")
        rgb.load()  # Fully decode now, so corrupt files fail during the audit.
        digest = hashlib.sha256()
        digest.update(str(rgb.size).encode("utf-8"))
        digest.update(rgb.tobytes())
        return digest.hexdigest()

fingerprints = []
broken = []
for path, target in catalog.samples:
    try:
        fingerprints.append(pixel_fingerprint(path))
    except (OSError, ValueError) as error:
        broken.append((path, str(error)))
        fingerprints.append(None)
if broken:
    display(pd.DataFrame(broken, columns=["path", "error"]).head(20))
    raise ValueError("Fix or remove the unreadable images from the dataset, then rerun.")

duplicate_groups = defaultdict(list)
for index, fingerprint in enumerate(fingerprints):
    duplicate_groups[fingerprint].append(index)
duplicate_groups = {key: value for key, value in duplicate_groups.items() if len(value) > 1}
print("Readable images:", len(catalog), "| Duplicate-content groups:", len(duplicate_groups))
conflicting = [ids for ids in duplicate_groups.values() if len(set(labels[ids])) > 1]
if conflicting:
    raise ValueError("Identical image content has conflicting labels. Review these groups: "
                     + str(conflicting[:5]))


## Step 5 · Create reproducible, stratified index splits

Reserve 20% for test and then 20% of the remainder for validation: approximately
64% training, 16% validation, and 20% test. Stratification preserves class coverage.
The file order and split seed form a reproducibility contract. Duplicate images
across splits trigger a clear failure in the next step; remove duplicates or keep
each duplicate/group together before using a real dataset.


In [ ]:
all_indices = np.arange(len(catalog))
try:
    train_val_idx, test_idx = train_test_split(
        all_indices, test_size=0.20, random_state=42, stratify=labels,
    )
    train_idx, val_idx = train_test_split(
        train_val_idx, test_size=0.20, random_state=42, stratify=labels[train_val_idx],
    )
except ValueError as error:
    raise ValueError("The split needs more images per class. Inspect the class counts.") from error

splits = {"train": train_idx, "validation": val_idx, "test": test_idx}
for name, indices in splits.items():
    assert set(labels[indices]) == set(range(len(class_names))), f"Missing class in {name}."
assert not (set(train_idx) & set(val_idx) or set(train_idx) & set(test_idx) or set(val_idx) & set(test_idx))
display(pd.DataFrame({
    name: np.bincount(labels[indices], minlength=len(class_names))
    for name, indices in splits.items()
}, index=class_names))


## Step 6 · Verify that content does not cross a split boundary

Index disjointness alone is insufficient: two different paths can contain the same
picture. Compare content fingerprints for every split pair. This check runs on the
full selected folder before any smoke-test reduction, so a tiny run cannot hide a
duplicate elsewhere in the dataset.


In [ ]:
split_hashes = {name: {fingerprints[int(i)] for i in indices} for name, indices in splits.items()}
leakage = {}
for left, right in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    shared = split_hashes[left] & split_hashes[right]
    if shared:
        leakage[f"{left} ↔ {right}"] = len(shared)
if leakage:
    raise ValueError(f"Duplicate image content crosses splits: {leakage}. "
                     "Deduplicate or split by content/group before training.")
print("Passed: split indices are disjoint and no exact decoded image crosses splits.")

# Keep all classes represented when reducing to a tiny pipeline check.
if SMOKE:
    splits = {
        name: np.concatenate([indices[labels[indices] == label][:4]
                              for label in range(len(class_names))])
        for name, indices in splits.items()
    }
train_idx, val_idx, test_idx = (splits[name] for name in ["train", "validation", "test"])


## Step 7 · Keep train and evaluation transformations separate

Each `Subset` points to its own `ImageFolder` view. Sharing one mutable dataset and
then changing `.transform` would also change the validation transform. Here training
gets a random flip; evaluation is deterministic. Review whether a flip preserves
your real labels before using it for letters, medical images, or directional objects.


In [ ]:
MEAN, STD = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)
train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
train_view = datasets.ImageFolder(DATASET_ROOT, transform=train_transform)
eval_view = datasets.ImageFolder(DATASET_ROOT, transform=eval_transform)
assert train_view.class_to_idx == eval_view.class_to_idx == catalog.class_to_idx
assert train_view.samples == eval_view.samples == catalog.samples
train_ds = Subset(train_view, train_idx.tolist())
val_ds = Subset(eval_view, val_idx.tolist())
test_ds = Subset(eval_view, test_idx.tolist())
train_dl = cu.loader(train_ds, batch_size=BATCH_SIZE, shuffle=True, seed=42)
val_dl = cu.loader(val_ds, batch_size=BATCH_SIZE)
test_dl = cu.loader(test_ds, batch_size=BATCH_SIZE)
print("Train / validation / test:", len(train_ds), len(val_ds), len(test_ds))
cu.show_images(val_ds, class_names, n=9, mean=MEAN, std=STD)


## Step 8 · Save a split manifest

An index alone stops being meaningful if files are added or reordered. A manifest
records relative paths, labels, and fingerprints for this run. It also makes the
demonstration auditable. The manifest is saved under `artifacts`, not in your dataset.
This cell replaces the notebook's previous manifest when you rerun it.


In [ ]:
dataset_root_resolved = Path(DATASET_ROOT).resolve()
manifest = {
    "seed": 42,
    "dataset_root": str(dataset_root_resolved),
    "class_to_idx": catalog.class_to_idx,
    "splits": {
        name: [{
            "path": str(Path(catalog.samples[int(i)][0]).resolve().relative_to(dataset_root_resolved)),
            "label": int(labels[i]), "pixel_sha256": fingerprints[int(i)],
        } for i in indices]
        for name, indices in splits.items()
    },
}
manifest_path = cu.ARTIFACTS / "07_split_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Saved:", manifest_path)


## Step 9 · Train a classifier matched to your class count

This compact CNN learns from scratch and accepts three-channel images. The final
layer has one output per folder. `fit` restores the epoch with the best validation
loss. Tiny synthetic datasets may produce noisy curves; use them to explain the
training mechanics, not to promise real-data accuracy.


In [ ]:
cu.seed_everything(42)
model = cu.SmallCNN(num_classes=len(class_names), in_channels=3).to(DEVICE)
history = cu.fit(model, train_dl, val_dl, epochs=EPOCHS, device=DEVICE, lr=1e-3)
cu.plot_history(history)
print("Selected-model validation:", cu.evaluate(model, val_dl, DEVICE))


## Step 10 · Evaluate once and inspect per-class errors

After fixing the training procedure, evaluate on test data. A confusion matrix shows
which class pairs are confused; overall accuracy alone can hide this. A class may
receive no predictions in a tiny run, so the report uses `zero_division=0` explicitly.


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

test_metrics = cu.evaluate(model, test_dl, DEVICE)
test_logits, test_labels = cu.predict(model, test_dl, DEVICE)
test_predictions = test_logits.argmax(1)
print("Held-out test:", test_metrics)
print(classification_report(test_labels.numpy(), test_predictions.numpy(),
                            labels=list(range(len(class_names))),
                            target_names=class_names, zero_division=0))
ConfusionMatrixDisplay.from_predictions(
    test_labels.numpy(), test_predictions.numpy(), labels=list(range(len(class_names))),
    display_labels=class_names, cmap="Blues", xticks_rotation=30,
)
plt.title("Held-out test confusion matrix")
plt.tight_layout()
plt.show()


## Step 11 · Package and reload a prediction

Save class names and the class-to-index map in the same artifact as the state dict.
Reload on a newly constructed model and classify one image using the deterministic
transform. Rebuilding a label map from a different folder at inference can silently
assign the wrong human-readable name to a correct numeric prediction.


In [ ]:
model_path = cu.ARTIFACTS / "07_custom_classifier.pt"
torch.save({
    "model_state": {k: v.detach().cpu() for k, v in model.state_dict().items()},
    "architecture": "SmallCNN", "in_channels": 3,
    "class_names": list(class_names), "class_to_idx": catalog.class_to_idx,
    "image_size": [64, 64], "mean": list(MEAN), "std": list(STD),
}, model_path)
saved = torch.load(model_path, map_location="cpu", weights_only=True)
loaded = cu.SmallCNN(num_classes=len(saved["class_names"]), in_channels=3).to(DEVICE)
loaded.load_state_dict(saved["model_state"])
loaded.eval()

image_path = Path(catalog.samples[int(test_idx[0])][0])
with Image.open(image_path) as source:
    image = source.convert("RGB")
with torch.inference_mode():
    probability = loaded(eval_transform(image).unsqueeze(0).to(DEVICE)).softmax(1)[0].cpu()
predicted_id = probability.argmax().item()
print("Image:", image_path.name)
print("Prediction:", saved["class_names"][predicted_id], "| softmax score:", float(probability.max()))
plt.imshow(image)
plt.axis("off")
plt.show()
print("Saved:", model_path)


## Step 12 · Demonstration challenges

1. For the default synthetic dataset, create a **separate shifted test view** that
   recolors background pixels, using the example below. Keep the trained model fixed
   and select the new color before measuring. Does original validation performance
   predict performance on these shifted images? Editing `generate_shapes` and rerunning
   will not change existing files because the generator skips existing filenames.
2. In a copy of your dataset, duplicate an image into a different class. Which audit
   stops the run, and why should it stop before training?
3. Adapt this pipeline to your own two-class collection. Inspect every class mapping,
   remove duplicate images, and record any patient/session/product grouping.
4. Replace this CNN with notebook 06's pretrained model. Use its preprocessing and
   compare **validation** results before opening a new final test set.

<details><summary>Hints / solutions</summary>

Background changes create a distribution shift. This wrapper creates shifted examples
in memory and leaves both the original dataset and model unchanged. It applies only
to our synthetic shapes: their top-left pixel is always background, and background
and foreground color ranges do not overlap. It is not a segmentation method for
photographs. The tensors use this notebook's `Normalize(0.5, 0.5)` preprocessing.

```python
class BackgroundShift(torch.utils.data.Dataset):
    def __init__(self, base_dataset, rgb=(0.25, 0.35, 0.45)):
        self.base_dataset = base_dataset
        self.new_color = 2 * torch.tensor(rgb, dtype=torch.float32) - 1

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, index):
        image, label = self.base_dataset[index]
        background = image[:, 0, 0]
        mask = (image - background[:, None, None]).abs().amax(dim=0) < 1e-6
        shifted = image.clone()
        shifted[:, mask] = self.new_color[:, None]
        return shifted, label

assert Path(DATASET_ROOT).resolve() == GENERATED_ROOT.resolve()
shifted_test_ds = BackgroundShift(test_ds)
cu.show_images(shifted_test_ds, class_names, n=9, mean=MEAN, std=STD)
print(cu.evaluate(model, cu.loader(shifted_test_ds), DEVICE))
```

Predeclare this study's color; choosing the color or retraining after inspecting its
test outcomes needs a fresh final evaluation set. The paired shifted examples measure
this specific intervention, not performance on independently collected photographs.
Identical pixels with different labels trigger the
conflicting-label audit. Exact content shared across splits triggers the leakage
audit. For grouped data, use group-aware splits (for example `GroupShuffleSplit`)
and then verify class coverage. A class count is not enough to detect group leakage.

```python
# Inspect the saved labels before deploying:
print(saved["class_to_idx"])
assert saved["class_names"][0] == min(saved["class_to_idx"], key=saved["class_to_idx"].get)
```

</details>

**References:** [torchvision ImageFolder](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.ImageFolder.html),
[PyTorch datasets and loaders](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html),
[scikit-learn stratified splitting](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html),
[group-aware splitting](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html).
